# Tutorial 5: Sentiment Analysis Dataset with a Manifest

Build a small sentiment-analysis dataset as LAILA entries, organise them
under a **`Manifest`**, push everything to S3, then remember and inspect
the dataset from scratch using only the manifest nickname.

Each datapoint is stored as two entries (`text_en_base` for the sentence,
`label_en_base` for the sentiment) grouped under a nested blueprint key
`datapoint_0` … `datapoint_24`.

**Prerequisites:** `pip install "laila-core[s3]"` and a `secrets.toml` with your AWS credentials.

In [ ]:
import laila
from laila.pool import S3Pool
from laila.policy.central.memory.schema import Manifest

In [ ]:
laila.read_args("./secrets.toml")

s3_pool = S3Pool(
    bucket_name=laila.args.AWS_BUCKET_NAME,
    access_key_id=laila.args.AWS_ACCESS_KEY_ID,
    secret_access_key=laila.args.AWS_SECRET_ACCESS_KEY,
    region_name=laila.args.AWS_REGION,
    nickname="sentiment_pool",
)
laila.memory.extend(s3_pool, pool_nickname="sentiment_pool")

## Step 1: Define the raw dataset

25 short sentences with `positive` or `negative` sentiment labels.

In [ ]:
RAW_DATASET = [
    ("The sunrise over the lake was absolutely breathtaking.", "positive"),
    ("I can't believe how rude the cashier was today.", "negative"),
    ("This homemade pasta is the best I have ever tasted.", "positive"),
    ("The flight was delayed by six hours with no explanation.", "negative"),
    ("My daughter took her first steps today and I cried happy tears.", "positive"),
    ("The hotel room smelled terrible and the sheets were stained.", "negative"),
    ("I got promoted at work after years of hard effort.", "positive"),
    ("The new software update broke half of my workflow.", "negative"),
    ("Our neighbourhood organised a wonderful block party yesterday.", "positive"),
    ("I waited forty minutes for cold, soggy fries.", "negative"),
    ("The book kept me up all night because the story was gripping.", "positive"),
    ("Customer support hung up on me twice in a row.", "negative"),
    ("The garden is blooming with colour after all the spring rain.", "positive"),
    ("My laptop crashed during the final exam and I lost my answers.", "negative"),
    ("Volunteering at the shelter was the most rewarding weekend.", "positive"),
    ("The parking garage charged me double without any receipt.", "negative"),
    ("We adopted the sweetest rescue dog from the local shelter.", "positive"),
    ("The dentist appointment was painful and they were running an hour late.", "negative"),
    ("The concert last night gave me chills from start to finish.", "positive"),
    ("My package arrived completely crushed and the item inside was broken.", "negative"),
    ("Learning to play the piano has been incredibly fulfilling.", "positive"),
    ("The movie was a predictable waste of two hours.", "negative"),
    ("I finally finished the marathon and beat my personal record.", "positive"),
    ("They cancelled my reservation without even notifying me.", "negative"),
    ("The kids surprised me with breakfast in bed for my birthday.", "positive")
]

print(f"Dataset size: {len(RAW_DATASET)} datapoints")
print(f"Positive: {sum(1 for _, l in RAW_DATASET if l == 'positive')}")
print(f"Negative: {sum(1 for _, l in RAW_DATASET if l == 'negative')}")

## Step 2: Wrap each datapoint as LAILA entries

For every sentence we create two constant entries — one for the text and
one for the label — then group them in a dict keyed `text_en_base` and
`label_en_base`.

In [ ]:
dataset_entries = {}

for idx, (text, label) in enumerate(RAW_DATASET):
    text_entry = laila.constant(data=text)
    label_entry = laila.constant(data=label)
    dataset_entries[f"datapoint_{idx}"] = {
        "text_en_base": text_entry,
        "label_en_base": label_entry,
    }

print(f"Created {len(dataset_entries)} datapoints, each with 2 entries")
print(f"Example key: 'datapoint_0'")
print(f"  text_en_base  -> {dataset_entries['datapoint_0']['text_en_base'].global_id}")
print(f"  label_en_base -> {dataset_entries['datapoint_0']['label_en_base'].global_id}")

## Step 3: Build the Manifest

The `Manifest` accepts the nested dict of Entry objects.  It automatically
extracts a *blueprint* (the same nested structure but with `global_id`
strings instead of Entry objects) and stashes the entries for upload.

In [ ]:
manifest = Manifest(
    data=dataset_entries,
    nickname="my_sentiment_dataset_v1",
)

print(f"Manifest global_id:  {manifest.global_id}")
print(f"Manifest is Entry:   {isinstance(manifest, laila.Entry)}")
print(f"Scope:               {manifest.scopes}")
print(f"Top-level keys:      {len(manifest.keys())}  (datapoint_0 … datapoint_24)")
print(f"Total leaf entries:  {sum(1 for _ in manifest)}")
print()
print("Blueprint preview (datapoint_0):")
for k, v in manifest.blueprint["datapoint_0"].items():
    print(f"  {k}: {v}")

## Step 4: Memorize everything to S3

`memorize()` uploads all 50 leaf entries **plus** the manifest's own
blueprint (as a MANIFEST-scoped entry) in one call.  It returns a
`GroupFuture` — wrap it in `laila.guarantee` to block until all writes finish.

In [ ]:
with laila.guarantee:
    manifest.memorize(pool_nickname="sentiment_pool")

print(f"Uploaded {sum(1 for _ in manifest)} entries + manifest blueprint to S3")

## Step 5: Destroy all local state

After this cell the only way to recover the dataset is through LAILA.

In [ ]:
manifest_nickname = "my_sentiment_dataset_v1"

del dataset_entries, manifest

print("All local state destroyed")

## Step 6: Remember the dataset from S3

1. Rebuild the manifest's identity from its nickname.
2. Remember the manifest's blueprint entry from S3.
3. Reconstruct the manifest with the remembered blueprint.
4. Use `remember()` to fetch every referenced entry.

In [ ]:
manifest = Manifest(nickname=manifest_nickname)

with laila.guarantee:
    ref = laila.remember(manifest.global_id, pool_nickname="sentiment_pool")
blueprint = ref.data[0]

manifest = Manifest(data=blueprint, nickname=manifest_nickname)

print(f"Blueprint remembered — {len(manifest.keys())} datapoints")
print(f"Leaf entries to remember: {sum(1 for _ in manifest)}")

In [ ]:
with laila.guarantee:
    remember_future = manifest.remember(pool_nickname="sentiment_pool")

remembered_data = remember_future.data
data_map = dict(zip(list(manifest), remembered_data))
print(f"Remembered {len(remembered_data)} entries from S3")

## Step 7: Inspect the remembered dataset

Walk the blueprint and print each datapoint using the entry map.

In [ ]:
print(f"{'#':<4} {'Sentiment':<10} Text")
print("-" * 72)

for i in range(len(manifest.keys())):
    dp = manifest.blueprint[f"datapoint_{i}"]
    text  = data_map[dp["text_en_base"]]
    label = data_map[dp["label_en_base"]]
    print(f"{i:<4} {label:<10} {text[:60]}")

In [ ]:
pos = sum(
    1 for i in range(len(manifest.keys()))
    if data_map[manifest.blueprint[f"datapoint_{i}"]["label_en_base"]] == "positive"
)
neg = len(manifest.keys()) - pos

print(f"Remembered dataset: {len(manifest.keys())} datapoints")
print(f"  positive: {pos}")
print(f"  negative: {neg}")

## Clean up

In [ ]:
with laila.guarantee:
    manifest.forget(pool_nickname="sentiment_pool")
print("All entries + manifest blueprint cleaned up from S3")

## Summary

- Each datapoint is two entries: `text_en_base` (the sentence) and `label_en_base` (the sentiment)
- The manifest blueprint mirrors the dataset structure: `datapoint_0` … `datapoint_24`, each a nested dict with those two keys
- **`manifest.memorize()`** pushes all 50 entries + the manifest itself to S3 in one call
- The manifest can be fully reconstructed from just its **nickname** — remember the blueprint, then `remember()` every referenced entry
- **`manifest.forget()`** deletes everything (entries + manifest) in one call
- This pattern generalises to any tabular or structured dataset — add columns, languages, or modalities as additional keys per datapoint